In [ ]:
import os
import sys
sys.path.insert(0,'/home/jietang/xenon_plot_style')
from xenonnt_plot_style import XENONPlotStyle as xps
xps.use('xenonnt')

import cutax
import strax
import straxen

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json
import time
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

from cutax.cut_lists.basic import BasicCuts,MinimalCuts
from cutax.cut_lists.kr83m import Kr83mSelection
from straxen.itp_map import InterpolatingMap
from load_data import load_s1_data

import getpass
user = getpass.getuser() #Figure out who is running this in order to set paths.


In [ ]:
import pandas as pd
# Set the maximum column width to None to show full content
pd.set_option('display.max_colwidth', None)  # Show full content in each column

# Optionally, you can also set max rows and columns if needed
pd.set_option('display.max_rows', 100)  # Show all rows
pd.set_option('display.max_columns', 10)  # Show all columns
pd.set_option('display.expand_frame_repr', False)  # Prevent wrapping to a new line


In [ ]:
straxen.print_versions()


In [ ]:
st = straxen.contexts.xenonnt_online(output_folder = '/scratch/midway3/jietang/strax_data')
st.storage.append(strax.DataDirectory('/project/lgrandi-shared/commissioning/',readonly=True))

# ═══════════════════════════════════════════════════════════════
# Register naive_peak_positions plugin (embedded inline)
# ═══════════════════════════════════════════════════════════════
# This plugin computes x,y from area_per_channel weighted by PMT positions.
# strax auto-loads 'peaks' as a dependency — we never write 'peaks' explicitly.

class PeakPositionsWeighted(strax.Plugin):
    """
    Provides naive XY position estimates based on area-weighing.
    """
    provides = 'peak_positions_weighted'
    depends_on = ('peaks',)
    dtype = [('x_weighted', np.float32, 'Reconstructed X position (cm), uncorrected'),
             ('y_weighted', np.float32, 'Reconstructed Y position (cm), uncorrected')] + strax.time_fields
    __version__ = '0.0.7'
 
    def compute(self, peaks):
        result = np.ones(len(peaks), dtype=self.dtype)
        result['time'], result['endtime'] = peaks['time'], strax.endtime(peaks)
 
        result['x_weighted'] *= float('nan')
        result['y_weighted'] *= float('nan')
 
        # weigh PMT positions with absoulte area observed in corresponding PMT
        area_per_channel = peaks['area_per_channel']
        weight_map = np.abs(area_per_channel) / np.abs(area_per_channel).sum(axis=1)[:, None]
 
        result['x_weighted'] = (weight_map * straxen.pmt_positions().x.values).sum(axis=1)
        result['y_weighted'] = (weight_map * straxen.pmt_positions().y.values).sum(axis=1)
 
        return result


In [ ]:
st.data_info("peak_basics")


In [ ]:
run_mode = "tpc_bkg",
exclude_tags = "abandon", "messy"
include_tags = "s1_only"

targets = ["peak_basics"]


In [ ]:
runs_sr3 = st.select_runs(run_mode=run_mode, 
                          exclude_tags=exclude_tags, 
                          include_tags=include_tags,
                          available=('peak_basics'), 
                                    )


In [ ]:
runs_sr3


In [ ]:
# ============================================================
# Run selection: newest ~300 s1_only runs available
# ============================================================

# Convert 'name' to integer for sorting
runs_sr3['name_int'] = runs_sr3['name'].astype(int)

# Sort by run number descending (newest first), take newest 300
runs_sorted = runs_sr3.sort_values('name_int', ascending=False)
runs_newest = runs_sorted.head(300)

run_id_sr3 = runs_newest['name'].tolist()
run_id_sr3 = run_id_sr3[::-1]  # Reverse to go oldest→newest for loading

print(f"Total s1_only runs available: {len(runs_sr3)}")
print(f"Selected newest {len(run_id_sr3)} runs")
print(f"Run range: {run_id_sr3[0]} — {run_id_sr3[-1]}")
print(f"First 5: {run_id_sr3[:5]}")
print(f"Last 5:  {run_id_sr3[-5:]}")


In [ ]:
# ============================================================
# columns_to_keep — what we load from peak_basics
# ============================================================
# Only peak_basics is available (peaks/peaklets not stored).
# No position info (x, y, r2) — radial cuts are not possible.

columns_to_keep = [
    'time',              # time axis for rate evolution
    'endtime',           # livetime bookkeeping
    'area',              # THE observable — alpha peak identification
    'area_fraction_top', # z-proxy for AFT correction
    'range_50p_area',    # alpha ROI cut (width cut)
    'type',              # peak classification (filter type==1 post-load)
]

print(f"peak_basics columns to load ({len(columns_to_keep)}):")
for c in columns_to_keep:
    print(f"  {c}")
print("\nNote: peaks/peaklets not available — no x, y, r2 for radial cuts.")


In [ ]:
# Define ROI:
alpha_roi_area = [5e3, 150e3]       # PE — alphas live here
alpha_roi_50p_width = [40, 200]     # ns — S1-like pulse shape

# Build the strax selection string (applied DURING loading → saves RAM)
cut_str = (
    f"(area > {alpha_roi_area[0]}) & "
    f"(area < {alpha_roi_area[1]}) & "
    f"(range_50p_area > {alpha_roi_50p_width[0]}) & "
    f"(range_50p_area < {alpha_roi_50p_width[1]}) & "
    f"(type == 1)"
)


In [ ]:
all_runs = run_id_sr3
frames = []

for this_run in tqdm(all_runs):
    this_run_name = str(this_run)
    
    if not st.is_stored(this_run_name, "peak_basics"):
        continue
    
    try:
        run_df = st.get_df(
            this_run_name,
            'peak_basics',
            progress_bar=False,
        )
    except Exception as e:
        print(f"Failed on {this_run_name}: {e}")
        continue
    
    if len(run_df) == 0:
        continue
    
    available_cols = [c for c in columns_to_keep if c in run_df.columns]
    run_df = run_df[available_cols].copy()
    run_df['run_name'] = this_run_name
    
    frames.append(run_df)

df_peaks = pd.concat(frames, ignore_index=True)
print(f"Total peaks loaded: {len(df_peaks)}")
print(f"Columns: {df_peaks.columns.tolist()}")


In [ ]:
# ============================================================
# DATA LOADING — peak_basics only (peaks/peaklets not available)
# ============================================================
# Since peaks is not stored, we can only use peak_basics.
# No position info (x, y, r2) — Cut 4 (r² FV) will be skipped.
# Filtering is done post-load.

all_runs = run_id_sr3

frames = []
runs_loaded = 0
runs_failed = 0

for this_run in tqdm(all_runs):
    this_run_name = this_run if isinstance(this_run, str) else f"{this_run:06d}"
    
    if not st.is_stored(this_run_name, "peak_basics"):
        continue
    
    try:
        run_df = st.get_df(
            this_run_name,
            'peak_basics',
            progress_bar=False,
        )
        runs_loaded += 1
    except Exception as e:
        runs_failed += 1
        if runs_failed <= 3:
            print(f"Failed on {this_run_name}: {e}")
        continue
    
    if len(run_df) == 0:
        continue
    
    available_cols = [c for c in columns_to_keep if c in run_df.columns]
    run_df = run_df[available_cols].copy()
    run_df['run_name'] = this_run_name
    frames.append(run_df)

print(f"\nRuns loaded: {runs_loaded}, failed: {runs_failed}")

if frames:
    df = pd.concat(frames, ignore_index=True)
    print(f"Total candidates (all types): {len(df):,}")
    
    # ── Post-load filtering ──
    n_before = len(df)
    
    # 1. Type: keep only S1 (type==1)
    df = df[df['type'] == 1]
    print(f"  After type==1: {len(df):,}  (removed {n_before - len(df):,})")
    
    # 2. Area ROI
    mask_area = (df['area'] >= alpha_roi_area[0]) & (df['area'] <= alpha_roi_area[1])
    df = df[mask_area]
    
    # 3. Pulse width ROI
    mask_width = (df['range_50p_area'] >= alpha_roi_50p_width[0]) & (df['range_50p_area'] <= alpha_roi_50p_width[1])
    df = df[mask_width]
    
    print(f"  After area + width ROI: {len(df):,} S1 alpha candidates")
    print(f"\nColumns loaded: {list(df.columns)}")
    
    del frames
else:
    print("ERROR: No data loaded!")
    df = None


In [ ]:
# Save for next time so you don't have to reload
df.to_pickle("/project2/lgrandi/jietang/sr3_commissioning/sr3_alpha_s1only.pkl")
# To reload later:
# df = pd.read_pickle("data/sr3_alpha_candidates.pkl")


In [ ]:
_data = df_peaks if "df_peaks" in globals() else df

fig, ax = plt.subplots(figsize=(6,4))
h = ax.hist2d(
    _data['area'], 
    _data['area_fraction_top'],
    bins=[200, 200],
    range=[[2e3, 100e3], [0.05, 0.6]],
    cmin=1,
    norm=matplotlib.colors.LogNorm()
)
plt.colorbar(h[3], ax=ax, label='Counts')
ax.set_xlabel('Area [PE]')
ax.set_ylabel('Area Fraction Top')
ax.set_title('Area vs AFT')
plt.show()


## S1-Only Alpha Cuts — Per-Cut Tuning

Apply cuts **before the AFT correction** to clean the band for better resolution.

Each cell below is self-contained: **parameters at the top**, diagnostic plot, pass/fail stats.
Tune parameters, re-run the cell, see the effect immediately.
Run **Apply All Cuts** when satisfied — this produces `df_cut` used in the AFT correction below.

| # | Cut | What it removes | Why it helps |
|---|-----|-----------------|--------------|
| 1 | AFT lower bound | Bottom-wall BiPos | Cleans the low-AFT tail of the band |
| 2 | AFT upper bound | Gas band near liquid surface | Removes outliers at high AFT |
| 3 | Width refinement | Non-alpha S1 pulses | Tightens the S1 pulse shape selection |
| 4 | r² fiducial volume *(skipped — no positions)* | Events near TPC walls | (requires x,y from peaks → N/A) |

**Note:** peaks/peaklets not available → no x, y, r² columns. Cut 4 will auto-skip.

---


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CUT 1: AFT Lower Bound — BiPo Rejection                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# Rejects events near the bottom PMT array wall where BiPo decays sit.
# These have low Area Fraction Top (light collected mostly by bottom PMTs).

# ── Tunable Parameter ──
C1_AFT_MIN = 0.12  # minimum AFT (rejects bottom wall BiPo events)

# Initialize shared mask dict
if "cut_masks" not in globals():
    cut_masks = {}

# Determine which DataFrame to use
_data = df_peaks if "df_peaks" in globals() else df

if _data is not None and 'area_fraction_top' in _data.columns:
    m1 = (_data['area_fraction_top'] > C1_AFT_MIN)
    cut_masks['aft_lower'] = m1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

    ax1.hist(_data['area_fraction_top'], bins=200, range=[0, 0.7],
             histtype='step', color='gray', lw=1, label=f'All ({len(_data):,})')
    ax1.hist(_data.loc[m1, 'area_fraction_top'], bins=200, range=[0, 0.7],
             histtype='stepfilled', color='steelblue', alpha=0.5,
             label=f'AFT > {C1_AFT_MIN} ({m1.sum():,})')
    ax1.axvline(x=C1_AFT_MIN, color='red', ls='--', lw=2)
    ax1.axvspan(0, C1_AFT_MIN, alpha=0.1, color='red')
    ax1.set_xlabel('Area Fraction Top')
    ax1.set_ylabel('Counts')
    ax1.set_yscale('log')
    ax1.set_title('Cut 1: AFT Distribution', fontweight='bold')
    ax1.legend(fontsize=8)

    h = ax2.hist2d(_data['area'] / 1e3, _data['area_fraction_top'],
                   bins=[200, 150], range=[[5, 110], [0, 0.7]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    ax2.axhline(y=C1_AFT_MIN, color='red', ls='--', lw=2)
    ax2.set_xlabel('Area [10$^3$ PE]')
    ax2.set_ylabel('Area Fraction Top')
    ax2.set_title('Area vs AFT')
    plt.colorbar(h[3], ax=ax2, label='Counts')

    plt.tight_layout()
    plt.show()

    n_pass = m1.sum()
    print(f"Cut 1 (AFT lower): {n_pass:,} / {len(_data):,} pass ({100*n_pass/len(_data):.1f}%)")
    print(f"  Removed {(~m1).sum():,} events (bottom wall BiPos)")
else:
    print("ERROR: data not available or area_fraction_top missing")
    cut_masks['aft_lower'] = pd.Series(True, index=_data.index) if _data is not None else None



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CUT 2: AFT Upper Bound — Gas Band Rejection                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# Removes events near the liquid surface (high AFT) where gas accumulates.

# ── Tunable Parameter ──
C2_AFT_MAX = 0.55  # maximum AFT

if "cut_masks" not in globals():
    cut_masks = {}
_data = df_peaks if "df_peaks" in globals() else df

if _data is not None and 'area_fraction_top' in _data.columns:
    m2 = (_data['area_fraction_top'] < C2_AFT_MAX)
    cut_masks['aft_upper'] = m2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

    ax1.hist(_data['area_fraction_top'], bins=200, range=[0, 0.7],
             histtype='step', color='gray', lw=1, label=f'All ({len(_data):,})')
    ax1.hist(_data.loc[m2, 'area_fraction_top'], bins=200, range=[0, 0.7],
             histtype='stepfilled', color='steelblue', alpha=0.5,
             label=f'AFT < {C2_AFT_MAX} ({m2.sum():,})')
    ax1.axvline(x=C2_AFT_MAX, color='red', ls='--', lw=2)
    ax1.axvspan(C2_AFT_MAX, 0.7, alpha=0.1, color='red')
    ax1.set_xlabel('Area Fraction Top')
    ax1.set_ylabel('Counts')
    ax1.set_yscale('log')
    ax1.set_title('Cut 2: AFT Upper Bound', fontweight='bold')
    ax1.legend(fontsize=8)

    h = ax2.hist2d(_data['area'] / 1e3, _data['area_fraction_top'],
                   bins=[200, 150], range=[[5, 110], [0, 0.7]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    ax2.axhline(y=C2_AFT_MAX, color='red', ls='--', lw=2)
    ax2.set_xlabel('Area [10$^3$ PE]')
    ax2.set_ylabel('Area Fraction Top')
    ax2.set_title('Area vs AFT')
    plt.colorbar(h[3], ax=ax2, label='Counts')

    plt.tight_layout()
    plt.show()

    n_pass = m2.sum()
    print(f"Cut 2 (AFT upper): {n_pass:,} / {len(_data):,} pass ({100*n_pass/len(_data):.1f}%)")
    print(f"  Removed {(~m2).sum():,} events (gas band)")
else:
    print("ERROR: data not available")
    cut_masks['aft_upper'] = pd.Series(True, index=_data.index) if _data is not None else None



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CUT 3: Width Refinement — range_50p_area                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# Tightens the S1 pulse width cut. Alpha S1 pulses have width ~70-150 ns.

# ── Tunable Parameters ──
C3_WIDTH_MIN = 40   # ns
C3_WIDTH_MAX = 200  # ns

if "cut_masks" not in globals():
    cut_masks = {}
_data = df_peaks if "df_peaks" in globals() else df

if _data is not None and 'range_50p_area' in _data.columns:
    m3 = (_data['range_50p_area'] > C3_WIDTH_MIN) & (_data['range_50p_area'] < C3_WIDTH_MAX)
    cut_masks['width'] = m3

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

    ax1.hist(_data['range_50p_area'], bins=200, range=[0, 500],
             histtype='step', color='gray', lw=1, label=f'All ({len(_data):,})')
    ax1.hist(_data.loc[m3, 'range_50p_area'], bins=200, range=[0, 500],
             histtype='stepfilled', color='steelblue', alpha=0.5,
             label=f'In range ({m3.sum():,})')
    ax1.axvline(x=C3_WIDTH_MIN, color='red', ls='--', lw=2)
    ax1.axvline(x=C3_WIDTH_MAX, color='red', ls='--', lw=2)
    ax1.set_xlabel('range_50p_area [ns]')
    ax1.set_ylabel('Counts')
    ax1.set_yscale('log')
    ax1.set_title('Cut 3: S1 Width Spectrum', fontweight='bold')
    ax1.legend(fontsize=8)

    h = ax2.hist2d(_data['area'] / 1e3, _data['range_50p_area'],
                   bins=[200, 150], range=[[5, 110], [0, 300]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    ax2.axhline(y=C3_WIDTH_MIN, color='red', ls='--', lw=2)
    ax2.axhline(y=C3_WIDTH_MAX, color='red', ls='--', lw=2)
    ax2.set_xlabel('Area [10$^3$ PE]')
    ax2.set_ylabel('range_50p_area [ns]')
    ax2.set_title('Area vs Width')
    plt.colorbar(h[3], ax=ax2, label='Counts')

    plt.tight_layout()
    plt.show()

    n_pass = m3.sum()
    print(f"Cut 3 (Width): {n_pass:,} / {len(_data):,} pass ({100*n_pass/len(_data):.1f}%)")
    print(f"  Removed {(~m3).sum():,} events")
else:
    print("ERROR: data not available or range_50p_area missing")
    cut_masks['width'] = pd.Series(True, index=_data.index) if _data is not None else None



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CUT 4: r² Fiducial Volume — SKIPPED (no position data available)           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# peaks/peaklets not available → no x, y, r² columns in peak_basics.
# This cut is auto-skipped (passes all events).

# ── Tunable Parameter (saved for when positions become available) ──
C4_R2_MAX = 3800.0  # cm² (r ≈ 61.6 cm, TPC R ≈ 66 cm)

if "cut_masks" not in globals():
    cut_masks = {}
_data = df_peaks if "df_peaks" in globals() else df

if _data is not None:
    has_r2 = 'r2' in _data.columns
    has_xy = 'x' in _data.columns and 'y' in _data.columns

    if has_r2 or has_xy:
        m4 = pd.Series(True, index=_data.index)
        if has_r2:
            m4 &= (_data['r2'] < C4_R2_MAX)
        else:
            m4 &= (_data['x']**2 + _data['y']**2 < C4_R2_MAX)
        cut_masks['r2_fv'] = m4

        # (plot code would go here — see S1S2 notebook for full version)
        fig, ax = plt.subplots(figsize=(6, 4))
        r2_col = 'r2' if has_r2 else None
        if r2_col:
            ax.hist(_data[r2_col], bins=200, range=[0, 6000],
                    histtype='step', color='gray', lw=1, label=f'All ({len(_data):,})')
            ax.hist(_data.loc[m4, r2_col], bins=200, range=[0, 6000],
                    histtype='stepfilled', color='steelblue', alpha=0.5,
                    label=f'r² < {C4_R2_MAX:.0f} ({m4.sum():,})')
            ax.axvline(x=C4_R2_MAX, color='red', ls='--', lw=2)
            ax.set_xlabel('r² [cm²]')
            ax.set_ylabel('Counts')
            ax.set_yscale('log')
            ax.set_title('Cut 4: r² Distribution', fontweight='bold')
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        n_pass = m4.sum()
        print(f"Cut 4 (r² FV): {n_pass:,} / {len(_data):,} pass ({100*n_pass/len(_data):.1f}%)")
    else:
        # No position columns available — skip this cut
        print("Cut 4 (r² FV): SKIPPED — no x, y, or r² columns in peak_basics.")
        print("  (peaks/peaklets not available, so positions cannot be computed.)")
        print(f"  Passing all {len(_data):,} events unchanged.")
        cut_masks['r2_fv'] = pd.Series(True, index=_data.index)
else:
    print("ERROR: data not available")
    cut_masks['r2_fv'] = None


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  APPLY ALL CUTS — combine masks → df_cut                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# Combines all cut masks and produces df_cut (cleaned data for AFT correction).

if "cut_masks" not in globals():
    cut_masks = {}
_data = df_peaks if "df_peaks" in globals() else df

if _data is not None:
    mask_combined = pd.Series(True, index=_data.index)
    for name, m in cut_masks.items():
        if m is not None:
            mask_combined &= m

    df_cut = _data[mask_combined].copy().reset_index(drop=True)
    n_before = len(_data)
    n_after = len(df_cut)

    print()
    print("=" * 70)
    print("  S1-ONLY ALPHA CUT FLOW")
    print("=" * 70)
    flow_data = []
    cum_mask = pd.Series(True, index=_data.index)
    cut_order = [
        ('aft_lower', '1. AFT lower bound (BiPo)'),
        ('aft_upper', '2. AFT upper bound (gas)'),
        ('width',     '3. Width refinement'),
        ('r2_fv',     '4. r² fiducial volume'),
    ]
    for key, label in cut_order:
        m = cut_masks.get(key)
        if m is not None:
            cum_mask &= m
        n_rem = cum_mask.sum()
        flow_data.append({'Step': label, 'Passing': f'{n_rem:,}', 'Efficiency [%]': f'{100*n_rem/n_before:.1f}'})

    flow_df = pd.DataFrame(flow_data)
    print(flow_df.to_string(index=False))
    print("-" * 70)
    print(f"  Before cuts: {n_before:,}")
    print(f"  After cuts:  {n_after:,}")
    print(f"  Efficiency:  {100*n_after/n_before:.1f}%")
    print("=" * 70)
    print(f"\ndf_cut has {len(df_cut):,} events — ready for AFT correction below.")
else:
    print("ERROR: No data available. Run data loading first.")
    df_cut = None



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Use df_cut from cuts section above (fallback to _data if not run)
_data = df_cut if 'df_cut' in globals() and df_cut is not None else df_peaks
print(f'AFT correction using _data: {len(_data):,} events')

# ============================================================
# STEP 1: Define the bump model (the function you fit against)
# ============================================================

def exp_bump_model(x, a, b, c, d, e):
    """
    Florian's empirical model for area vs. AFT dependence.
    
    x = AFT (area_fraction_top)
    a = amplitude
    b = shift
    c, d = exponential scales
    e = offset
    
    Returns the expected area for a given AFT value.
    """
    return a * np.exp((x + b) / c) * (1 - np.exp((x + b) / d)) + e


# ============================================================
# STEP 2: Prepare data — keep only "good" AFT region
# ============================================================

# Remove gas band (high AFT ~0.8) and extreme bottoms
df_fv = _data[
    (_data['area_fraction_top'] > 0.12) & 
    (_data['area_fraction_top'] < 0.65)
].copy()

print(f"Events in FV AFT range: {len(df_fv)}")


# ============================================================
# STEP 3: Manually define Po-214 band (brightest band)
# ============================================================
# Look at your area vs AFT 2D plot. Find the brightest (highest area) curved band.
# Define upper and lower boundaries that bracket ONLY this band.
# These parameters are for exp_bump_model(), not the fit result.

# Guesses for Po-214 band (adjust by eye after seeing the plot)
po214_upper_pars = [4.55e4, -4.88e-2, -0.2, -3.42e-2, 60e3]
po218_lower_pars = [4.55e4, -4.88e-2, -1.18e-1, -3.42e-2, 35e3]

# Tag events inside the Po-214 band
df_fv['in_po214'] = (
    (df_fv['area'] < exp_bump_model(df_fv['area_fraction_top'], *po214_upper_pars)) &
    (df_fv['area'] > exp_bump_model(df_fv['area_fraction_top'], *po218_lower_pars))
)

print(f"Events in Po-214 band: {df_fv['in_po214'].sum()}")


# ============================================================
# STEP 4: Plot to verify the band selection looks right
# ============================================================

fig, ax = plt.subplots(figsize=(8,6))

# Full 2D histogram
h = ax.hist2d(
    df_fv['area'], df_fv['area_fraction_top'],
    bins=[200, 200], 
    range=[[20e3, 100e3], [0.1, 0.65]],
    cmin=1, norm=matplotlib.colors.LogNorm()
)
plt.colorbar(h[3], ax=ax, label='Counts')

# Draw the two boundary curves
aft_fine = np.linspace(0.12, 0.65, 500)
ax.plot(exp_bump_model(aft_fine, *po214_upper_pars), aft_fine, 
        'r-', lw=2.5, label='Po-214 upper boundary')
ax.plot(exp_bump_model(aft_fine, *po218_lower_pars), aft_fine, 
        'orange', lw=2.5, label='Po-218 lower boundary')

ax.set_xlabel('Area [PE]')
ax.set_ylabel('Area Fraction Top')
ax.set_title('Po-214 Band Selection (orange band between curves)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.show()

# If the orange band doesn't look right, adjust the parameter lists above


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import matplotlib

# Use df_cut from cuts section above (fallback to _data if not run)
_data = df_cut if 'df_cut' in globals() and df_cut is not None else df_peaks
print(f'AFT correction using _data: {len(_data):,} events')

def exp_bump_model(x, a, b, c, d, e):
    """Florian's empirical model for area vs. AFT dependence."""
    return a * np.exp((x + b) / c) * (1 - np.exp((x + b) / d)) + e


# ============================================================
# STEP 1: Data prep — FV range only
# ============================================================

df_fv = _data[
    (_data['area_fraction_top'] >= 0.12) & 
    (_data['area_fraction_top'] <= 0.55)
].copy()

print(f"Events in FV AFT range [0.12, 0.55]: {len(df_fv)}")


# ============================================================
# STEP 2: Define manual boundaries for Rn-222/220 band
# ============================================================
# These are TEMPORARY guesses to select which events to fit.
# Adjust these parameters by eye after seeing the plot.

rn_upper_pars = [4.55e4, -4.88e-2, -0.1, -0.05, 40e3]
rn_lower_pars = [4.55e4, -4.88e-2, -0.09, -0.05, 34e3]

df_fv['in_rn_band'] = (
    (df_fv['area'] < exp_bump_model(df_fv['area_fraction_top'], *rn_upper_pars)) &
    (df_fv['area'] > exp_bump_model(df_fv['area_fraction_top'], *rn_lower_pars))
)

print(f"Events bracketed by Rn boundaries: {df_fv['in_rn_band'].sum()}")


# ============================================================
# STEP 3: Bin Rn events in AFT slices and get medians
# ============================================================

n_aft_bins = 50
aft_edges = np.linspace(0.12, 0.55, n_aft_bins + 1)
df_fv['aft_bin'] = pd.cut(df_fv['area_fraction_top'], bins=aft_edges)

medians_rn = (
    df_fv[df_fv['in_rn_band']]
    .groupby('aft_bin', observed=True)['area']
    .median()
)

aft_centers = medians_rn.index.map(lambda iv: iv.mid).astype(float).values
median_values = medians_rn.values

print(f"AFT bins with Rn data: {len(aft_centers)}")
print(f"Median area range: {median_values.min():.0f} - {median_values.max():.0f} PE")


# ============================================================
# STEP 4: Fit the bump model to Rn medians
# ============================================================

p0_rn = [4.55e4, -0.05, -0.1, -0.05, 35e3]
p_opt_rn, p_cov_rn = curve_fit(
    exp_bump_model,
    aft_centers,
    median_values,
    p0=p0_rn,
    bounds=([0, -np.inf, -np.inf, -np.inf, 0],
            [np.inf, 0, 0, 0, np.inf]),
    maxfev=10000
)

p_err_rn = np.sqrt(np.diag(p_cov_rn))

print("\n=== Rn-222/220 FIT RESULTS ===")
print(f"a (amplitude) = {p_opt_rn[0]:.2e} ± {p_err_rn[0]:.2e}")
print(f"b (shift)     = {p_opt_rn[1]:.6f} ± {p_err_rn[1]:.6f}")
print(f"c (scale1)    = {p_opt_rn[2]:.6f} ± {p_err_rn[2]:.6f}")
print(f"d (scale2)    = {p_opt_rn[3]:.6f} ± {p_err_rn[3]:.6f}")
print(f"e (offset)    = {p_opt_rn[4]:.2e} ± {p_err_rn[4]:.2e}")


# ============================================================
# STEP 5: Plot everything
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Full 2D histogram (all data in FV range)
h = ax.hist2d(
    df_fv['area'], df_fv['area_fraction_top'],
    bins=[300, 300],
    range=[[20e3, 100e3], [0.1, 0.6]],
    cmin=1, norm=matplotlib.colors.LogNorm()
)
cbar = plt.colorbar(h[3], ax=ax, label='Counts')

# Fine grid for plotting curves
aft_fine = np.linspace(0.10, 0.60, 500)

# DASHED lines: manual boundaries (temporary)
ax.plot(exp_bump_model(aft_fine, *rn_upper_pars), aft_fine,
        '--', color='cyan', lw=2, alpha=0.7, label='Rn upper boundary (manual)')
ax.plot(exp_bump_model(aft_fine, *rn_lower_pars), aft_fine,
        '--', color='cyan', lw=2, alpha=0.7, label='Rn lower boundary (manual)')

# SOLID line: fitted centerline
ax.plot(exp_bump_model(aft_fine, *p_opt_rn), aft_fine,
        '-', color='yellow', lw=3, label='Rn fitted centerline')

# Cyan dots: median points used in fit
ax.plot(median_values, aft_centers, 'o', color='lime', markersize=6,
        label=f'Medians ({len(aft_centers)} bins)', zorder=5)

# Annotations
ax.set_xlabel('Area [PE]', fontsize=12)
ax.set_ylabel('Area Fraction Top', fontsize=12)
ax.set_title('Rn-222/220 Band: Manual Boundaries (dashed) + Fitted Curve (solid)', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.2)

plt.show()


In [ ]:
# ============================================================
# STEP 6: Apply AFT correction to all events
# ============================================================

# Choose a reference AFT value (somewhere in the middle of your range)
aft_ref = 0.30

# Calculate the reference area at this AFT
area_ref_rn = exp_bump_model(aft_ref, *p_opt_rn)

print(f"\nReference AFT: {aft_ref}")
print(f"Reference area (Rn centerline at AFT={aft_ref}): {area_ref_rn:.0f} PE")

# Apply correction: normalize all areas to the reference AFT
# corrected_area = raw_area * (area_at_ref_AFT / area_at_observed_AFT)
df_fv['area_corrected_rn'] = (
    df_fv['area'] 
    * area_ref_rn 
    / exp_bump_model(df_fv['area_fraction_top'], *p_opt_rn)
)

print(f"Correction applied to {len(df_fv)} events")


# ============================================================
# STEP 7: Plot before/after correction
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# BEFORE correction
ax = axes[0]
h = ax.hist2d(
    df_fv['area'], df_fv['area_fraction_top'],
    bins=[300, 300],
    range=[[20e3, 100e3], [0.1, 0.6]],
    cmin=1, norm=matplotlib.colors.LogNorm()
)
plt.colorbar(h[3], ax=ax, label='Counts')
ax.plot(exp_bump_model(aft_fine, *p_opt_rn), aft_fine,
        '-', color='yellow', lw=2, label='Rn centerline')
ax.set_xlabel('Area [PE]', fontsize=12)
ax.set_ylabel('Area Fraction Top', fontsize=12)
ax.set_title('BEFORE AFT Correction', fontsize=13, pad=20)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# AFTER correction
ax = axes[1]
h = ax.hist2d(
    df_fv['area_corrected_rn'], df_fv['area_fraction_top'],
    bins=[300, 300],
    range=[[20e3, 100e3], [0.1, 0.6]],
    cmin=1, norm=matplotlib.colors.LogNorm()
)
plt.colorbar(h[3], ax=ax, label='Counts')
ax.axvline(x=area_ref_rn, color='yellow', lw=2, linestyle='-', label=f'Reference area = {area_ref_rn:.0f} PE')
ax.set_xlabel('Corrected Area [PE]', fontsize=12)
ax.set_ylabel('Area Fraction Top', fontsize=12)
ax.set_title('AFTER AFT Correction', fontsize=13, pad=20)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.show()
plt.savefig("rn_band_correction_before_after.png")


In [ ]:
# ============================================================
# STEP 9: Plot 1D corrected Rn spectrum for inspection
# ============================================================

# Use only Rn-corrected events (from Rn band)
rn_events = df_fv[df_fv['in_rn_band']].copy()

print(f"Rn-222/220 events for rate calculation: {len(rn_events)}")

# Plot 1D corrected spectrum
fig, ax = plt.subplots(figsize=(14, 6))

counts_rn, edges_rn, _ = ax.hist(
    rn_events['area_corrected_rn'], 
    bins=400, 
    range=[30e3, 50e3],
    histtype='step', 
    color='yellow', 
    lw=2.5,
    label='Rn band (corrected)'
)

ax.set_xlabel('Corrected Area [PE]', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_yscale('log')
ax.set_title('1D Corrected Rn Spectrum - Inspect to find peak positions', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print some stats
print(f"\nSpectrum range: {edges_rn[0]:.0f} - {edges_rn[-1]:.0f} PE")
print(f"Max bin count: {counts_rn.max()}")
print(f"Total counts: {counts_rn.sum()}")

# Find rough peak positions (local maxima)
from scipy.signal import find_peaks
peaks_idx, peaks_props = find_peaks(counts_rn, height=counts_rn.max()*0.1, distance=20)
peak_positions = edges_rn[peaks_idx]
peak_heights = counts_rn[peaks_idx]

print(f"\nRough peak positions detected:")
for pos, height in zip(peak_positions, peak_heights):
    print(f"  {pos:.0f} PE (height: {height:.0f})")


In [ ]:
# ============================================================
# STEP 10: Fit single Gaussian to Rn spectrum
# ============================================================

from scipy.optimize import curve_fit as scipy_curve_fit

def gaussian(x, a, mu, sig, bg):
    """Single Gaussian + constant background."""
    return a * np.exp(-0.5 * ((x - mu) / sig) ** 2) + bg

# Prepare data for fitting
bin_centers = 0.5 * (edges_rn[:-1] + edges_rn[1:])
mask_fit = counts_rn > 0

# Initial guesses from the plot you showed
p0_single = [
    500,      # amplitude
    41100,    # mean (from detected peak)
    2500,     # sigma (rough width)
    1.0       # background
]

try:
    p_fit, p_cov_fit = scipy_curve_fit(
        gaussian,
        bin_centers[mask_fit],
        counts_rn[mask_fit],
        p0=p0_single,
        maxfev=10000,
        sigma=np.sqrt(counts_rn[mask_fit] + 1)  # Poisson errors
    )
    p_err = np.sqrt(np.diag(p_cov_fit))
    
except Exception as e:
    print(f"Fit failed: {e}")
    p_fit = None

if p_fit is not None:
    a_fit, mu_fit, sig_fit, bg_fit = p_fit
    
    print("\n=== SINGLE GAUSSIAN FIT ===")
    print(f"Amplitude: {a_fit:.1f} ± {p_err[0]:.1f}")
    print(f"Mean (μ):  {mu_fit:.0f} ± {p_err[1]:.0f} PE")
    print(f"Sigma (σ): {sig_fit:.0f} ± {p_err[2]:.0f} PE")
    print(f"Background: {bg_fit:.1f} ± {p_err[3]:.1f}")
    
    # ============================================================
    # STEP 11: Plot fit result
    # ============================================================
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Data
    ax.hist(rn_events['area_corrected_rn'], bins=400, range=[30e3, 50e3],
            histtype='step', color='yellow', lw=2.5, label='Data')
    
    # Fitted curve
    x_fine = np.linspace(30e3, 50e3, 1000)
    ax.plot(x_fine, gaussian(x_fine, *p_fit), 'r-', lw=3, label='Gaussian fit')
    
    # Components
    ax.axhline(y=bg_fit, color='gray', ls='--', lw=2, label=f'Background = {bg_fit:.1f}')
    
    # ±3σ region
    sig_3 = 3 * sig_fit
    ax.axvline(mu_fit, color='red', ls='-', lw=1.5, alpha=0.7)
    ax.axvline(mu_fit - sig_3, color='green', ls='--', lw=1.5, alpha=0.7, label=f'μ ± 3σ')
    ax.axvline(mu_fit + sig_3, color='green', ls='--', lw=1.5, alpha=0.7)
    ax.axvspan(mu_fit - sig_3, mu_fit + sig_3, alpha=0.1, color='green')
    
    ax.set_xlabel('Corrected Area [PE]', fontsize=12)
    ax.set_ylabel('Counts', fontsize=12)
    ax.set_yscale('log')
    ax.set_title('Single Gaussian Fit to Rn-222/220 Spectrum', fontsize=13)
    ax.legend(fontsize=11, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # STEP 12: Calculate Rn RATE
    # ============================================================
    
    # Count events within ±3σ
    rn_lo = mu_fit - sig_3
    rn_hi = mu_fit + sig_3
    
    n_rn = ((rn_events['area_corrected_rn'] >= rn_lo) & 
            (rn_events['area_corrected_rn'] <= rn_hi)).sum()
    
    print(f"\n=== Rn RATE CALCULATION ===")
    print(f"Peak position: {mu_fit:.0f} PE")
    print(f"Peak width (σ): {sig_fit:.0f} PE")
    print(f"Counting window [μ-3σ, μ+3σ]: [{rn_lo:.0f}, {rn_hi:.0f}] PE")
    print(f"Events in counting window: {n_rn}")
    
    # Get livetime
    livetime_ns = df_fv['endtime'].max() - df_fv['time'].min()
    livetime_s = livetime_ns / 1e9
    livetime_h = livetime_s / 3600
    
    rate_hz = n_rn / livetime_s
    rate_err_hz = np.sqrt(n_rn) / livetime_s
    
    print(f"\nLivetime: {livetime_s:.1f} s ({livetime_h:.2f} hours)")
    print(f"Rn rate: {rate_hz:.4f} ± {rate_err_hz:.4f} Hz")
    print(f"Rn rate: {rate_hz*1e6:.2f} ± {rate_err_hz*1e6:.2f} μHz")


In [ ]:
_data = df_peaks if "df_peaks" in globals() else df

fig, ax = plt.subplots(figsize=(6,4))
h = ax.hist2d(
    _data['area'], 
    _data['range_50p_area'],
    bins=[200, 200],
    range=[[2e3, 100e3], [50, 250]],
    cmin=1,
    norm=matplotlib.colors.LogNorm()
)
plt.colorbar(h[3], ax=ax, label='Counts')
ax.set_xlabel('Area [PE]')
ax.set_ylabel('Width_50p_area [ns]')
ax.set_title('Area vs Width')
plt.show()
